# Radar Financeiro V6

Preparar a tabela inicial com uma linha por MCI avaliado.

Na implementação, usar o mínimo de variáveis intermediárias. As regras de elegibilidade entram no cálculo das flags. Tratar CPF, agência, conta e outros valores no passo em que forem usados. Acrescentar validações separadas apenas se houver uma necessidade concreta.


## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


Use os gerenciadores carregados abaixo para acessar as fontes do Radar. O `gerenciador_spark_v2.ipynb` fornece utilitários para a sessão Spark já iniciada, e o `gerenciador_db2_spark_v2.ipynb` fornece o conector para consultas ao DB2.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-massivo-v9-sprint1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("[RADAR V9] Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v2.ipynb


## 2. Parâmetros da execução

Definir a data de execução (`DT_EXEA`), a quantidade de ciclos e a janela de sincronização, do dia correspondente um mês antes da execução até a véspera.


In [ ]:
dt_exea = date.fromisoformat(str(obter_variavel_ambiente('HOJE'))[:10])

conector_db2 = criar_conector_db2_spark()

## 3. Entrada do MCI


### Filtro inicial pela data de sincronização — Público Base

Em `DB2GFP.TRAN_RLZD_INST_PCT`, selecionar os MCIs com transações de estado `CD_EST_TRAN_INST = 0` incluídas na janela de sincronização. O início da janela é inclusivo (um mês antes de `HOJE`) e o fim é exclusivo (`HOJE`). O Público Base contém um `CD_CLI_REF` por MCI.


In [ ]:
%%spark
from datetime import date

# Reutilizar a leitura na última sincronização e nas flags de elegibilidade.
transacoes_recentes = conector_db2.sql(f"""
    SELECT CD_CLI, TS_INCL_TRAN, CD_TIP_PSS, NR_CPF_CNPJ_TITR,
           NR_AG_TITR, CD_CT_TITR, NR_MCA_PCT_OPB, CD_PRD
    FROM DB2GFP.TRAN_RLZD_INST_PCT
    WHERE CD_CLI IS NOT NULL
      AND CD_EST_TRAN_INST = 0
      AND TS_INCL_TRAN >= TIMESTAMP(DATE('{dt_exea}') - 1 MONTH)
      AND TS_INCL_TRAN < TIMESTAMP(DATE('{dt_exea}'))
""").persist()

publico_base = transacoes_recentes.selectExpr("CAST(CD_CLI AS INT) AS CD_CLI_REF").distinct()


### Adicionar a última sincronização por MCI

Na mesma leitura do Público Base, calcular `MAX(TS_INCL_TRAN)` por MCI e registrar em `TS_INCL_TRAN_REF`.


### Identificar Agro por MCI

Consultar `tabela_agro.flag_agro` pelo MCI e registrar em `FLAG_AGRO`. O indicador é informativo.


### Identificar pessoa física

Na leitura de transações recentes, usar `CD_TIP_PSS = 1` para calcular `FLAG_PESSOA_FISICA`.


## 4. Identificação e contexto


### Identificar conta corrente BB única

Na leitura de transações recentes, apurar se há uma única conta BB elegível com agência e conta em formato compatível com a consulta do ciclo. Registrar em `FLAG_CONTA_CORRENTE_BB_UNICA`.


### Identificar CPF único

Na leitura de transações recentes, contar os CPFs distintos preenchidos por MCI. Registrar em `FLAG_CPF_UNICO` e preparar o CPF selecionado para a consulta de renda.


### Consultar renda presumida

Consultar `DB2DFE.REN_AVLD_PF` pelo CPF dos MCIs aprovados nas regras anteriores. Usar a renda mais recente, multiplicada pela quantidade de ciclos, em `VL_REN_PRES_REF`. Preencher `FLAG_RENDA_PRESUMIDA_DISPONIVEL` quando houver valor, inclusive zero.


### Consultar macroperfil

Consultar `DB2D1D.DVS_GRDR_FNCO_PF` pelo MCI após a regra de renda e guardar o código mais recente até a execução em `CD_MAC_PRFL_CLI_REF`. Os códigos 1, 2 e 3 passam na regra de `FLAG_MACROPERFIL_VALIDO`.


### Consultar o dia do ciclo financeiro

Após as regras de elegibilidade, consultar `DB2GFP.CT_GRDR_FNCO` por agência e conta. Guardar o dia cadastrado mais recente em `DD_INC_MM_CLC_BLC_REF`, com valor nulo quando o cadastro não informar o dia.


## 5. Montagem da tabela inicial

Reunir os 12 campos do DDL em uma linha por MCI avaliado. Manter nulas as flags de elegibilidade das etapas que o MCI ainda não alcançou.
